<a href="https://colab.research.google.com/github/Shamsfathalla/FlyRank-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shamsfathalla/FlyRank-Starter-Notebooks/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose a Random Forest Classifier. It fits this lane because the relationship between search position and CTR is non-linear (as shown in the signal audit clicks drop heavily after position 3). A tree-based model handles these sharp drop-offs naturally without needing complex math adjustments.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
table_path = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix
import os

q = f"""
    SELECT
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        ga4_engaged_sessions,
        scroll_events
    FROM {table_path}
    WHERE month = '2026-03'
      AND gsc_impressions IS NOT NULL
    LIMIT 100000
"""
df = con.sql(q).df()

# Define the target proxy: 1 if missed opportunity (high impressions, 0 clicks), else 0
df['is_opportunity'] = ((df['gsc_impressions'] > 50) & (df['gsc_clicks'] == 0)).astype(int)

print(f"Data loaded. Total rows: {len(df)}")
print(f"Target distribution:\n{df['is_opportunity'].value_counts()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data loaded. Total rows: 100000
Target distribution:
is_opportunity
0    90311
1     9689
Name: count, dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I am splitting the data by content_hash_id (80% train, 20% test). This split is honest because it ensures the same page does not appear in both the training and test sets. We need to prove the model can score new pages it has never seen before, not just memorize the ones it already knows.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

unique_pages = df['content_hash_id'].unique()
train_pages, test_pages = train_test_split(unique_pages, test_size=0.2, random_state=42)

train_df = df[df['content_hash_id'].isin(train_pages)].copy()
test_df = df[df['content_hash_id'].isin(test_pages)].copy()

# Features (excluding clicks to prevent target leakage!)
features = ['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'scroll_events']
target = 'is_opportunity'

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

print(f"Training on {len(X_train)} rows.")
print(f"Testing on {len(X_test)} rows.")

Training on 80005 rows.
Testing on 19995 rows.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I am using ROC-AUC to compare. The baseline score from last week is gsc_impressions - gsc_clicks. The ML model uses the 5 core features without looking at clicks. The model attempts to find the underlying pattern of a "missed opportunity" rather than just using a fixed math formula.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

rf_model = RandomForestClassifier(max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

test_df['ml_score'] = rf_model.predict_proba(X_test)[:, 1]

test_df['baseline_score'] = test_df['gsc_impressions'] - test_df['gsc_clicks']

ml_auc = roc_auc_score(y_test, test_df['ml_score'])
baseline_auc = roc_auc_score(y_test, test_df['baseline_score'])

print("Model vs Baseline Comparison (ROC-AUC)")
print(f"Baseline Score Rule : {baseline_auc:.4f}")
print(f"Random Forest Model : {ml_auc:.4f}")

Model vs Baseline Comparison (ROC-AUC)
Baseline Score Rule : 0.9627
Random Forest Model : 0.9898


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model relies heavily on gsc_impressions and gsc_avg_position, which makes logical sense. When the model makes a mistake (False Positive), it flags a page with high impressions and a good position that is getting clicks, but maybe not enough to cross the proxy threshold. It struggles to know if a user's search intent was actually satisfied directly on Google.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

importances = pd.DataFrame({
    'Feature': features,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("Feature Importance")
print(importances.to_string(index=False))

# 0.5 threshold to see raw classification errors
y_pred = (test_df['ml_score'] > 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix")
print(f"True Negatives: {cm[0][0]} | False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}  | True Positives: {cm[1][1]}")

Feature Importance
             Feature  Importance
     gsc_impressions    0.816621
    gsc_avg_position    0.132326
        ga4_sessions    0.038583
       scroll_events    0.009776
ga4_engaged_sessions    0.002694

Confusion Matrix
True Negatives: 17547 | False Positives: 511
False Negatives: 156  | True Positives: 1781


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.